# Orbit Wars — PPO Self-Play on Kaggle GPU (v2)

**v2 changes vs the first run** (motivated by the slow learning curve in the first log — entropy stuck near max, 0% vs starter for 200 iters):

* **Reward shaping**: per-step `(Δmy_ships − Δopp_ships) / 50 × coef` gives a *dense* signal so we don't rely solely on sparse +1/−1 at termination. Coefficient decays toward 0 over training (anneals to pure sparse).
* **BC warm-start**: at startup, if no `last.pt` is found but `bc_init.pt` is (from `bc_pretrain_kaggle.ipynb`), load it as a starting point. Skips the random-exploration phase entirely.
* **Lower entropy bonus** (0.01 → 0.003): the v1 run had entropy stuck at near-max, meaning the policy never committed. Lower entropy coefficient lets PPO actually concentrate probability mass.
* **Higher LR** (3e-4 → 5e-4) + explicit LR override on resume so loaded optimizer state doesn't carry the old LR.
* **More games per iter** (8 → 12): less noisy gradient.

## How to use

1. **First time** running this: optionally run `bc_pretrain_kaggle.ipynb` first to produce `bc_init.pt` (~30 min, supervised on top-bot replays). This is highly recommended — it cuts the random-exploration phase from ~5K games to zero.
2. Attach an accelerator (GPU T4 ×2 — though the second T4 is idle, see notes below).
3. Save Version → Save & Run All. Kaggle persists `/kaggle/working/` between commits.
4. Next session, re-run — training auto-loads `last.pt` (or `bc_init.pt` if no `last.pt`) and continues.
5. When ready to deploy, run the **Export** cell — produces `weights.npz` + a torch-free `main_rl.py`.

## Why only one GPU is used

The network is tiny (~94 K params) — inference is bottlenecked by the CPU game-stepping loop (kaggle_environments runs pure-Python game logic). The second T4 sits idle. Real parallelism would require multiprocessing-based parallel workers — slated for a future revision; not essential to first reach ladder-competitive performance.

## Architecture

* **State encoder**: polar coordinates per planet + per fleet, masked transformer over variable-count entity sets.
* **Policy**: per my-planet pointer attention over target candidates + 4-bin discrete send fraction.
* **Value**: pooled global state → scalar.
* **Self-play league**: opponent sampled from past 8 snapshots, weighted by recency.
* **PPO**: clipped objective (ε=0.2), GAE (γ=0.99, λ=0.95), 4 epochs per batch, gradient clipping at 0.5.


## 1. Setup

In [ ]:
import sys, subprocess
# Kaggle base image already has torch + numpy; install kaggle-environments
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle-environments>=1.28.0'])
print('installed')

In [ ]:
import os, json, math, time, random, gc, glob
from collections import defaultdict, deque, namedtuple
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from kaggle_environments import make

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, mem={torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
LAST_CKPT = os.path.join(WORKING_DIR, 'last.pt')
BEST_CKPT = os.path.join(WORKING_DIR, 'best.pt')
LOG_FILE  = os.path.join(WORKING_DIR, 'train_log.jsonl')
print(f'checkpoint dir: {WORKING_DIR}')

## 2. Game constants & state encoder

Mirror the engine constants. State encoder converts obs → tensors.  
Polar coordinates `(r, θ)` are a better representation than `(x, y)` for orbital geometry.

In [ ]:
# Engine constants (match orbit_wars.py exactly)
BOARD = 100.0
CENTER = 50.0
SUN_R = 10.0
ROTATION_LIMIT = 50.0
MAX_SPEED = 6.0
TOTAL_STEPS = 500
MAX_PLANETS = 48     # 5-10 symmetric groups × 4 + up to 5×4 comets
MAX_FLEETS = 64

# Feature dimensions
PLANET_FEATURES = 12  # r, theta, dtheta_dt, radius, ships/100, prod/5, owner4hot(4), is_static, is_comet
FLEET_FEATURES = 8    # r, theta, angle_cos, angle_sin, ships/100, speed, owner3hot(3) — but use 2 (me/opp)
GLOBAL_FEATURES = 8   # step/500, ang_vel/0.05, my_ships/total, opp_ships/total, my_planets/N, opp_planets/N, comet_visible, mid_game_flag

SEND_FRACTIONS = [0.25, 0.5, 0.75, 1.0]  # 4 bins — 0 means NOOP (different head)
N_SEND_BINS = len(SEND_FRACTIONS)


def fleet_speed(ships):
    if ships <= 1: return 1.0
    r = min(1.0, max(0.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (r ** 1.5)


def encode_state(obs, player_id):
    """Convert raw obs → (planet_tensor, fleet_tensor, global_tensor, planet_mask, fleet_mask, my_planet_mask).
    All tensors shaped for batched processing. Owner re-mapped so player_id always = 0.
    Returns torch tensors on CPU (caller .to(device))."""
    raw_planets = obs.get('planets', []) if isinstance(obs, dict) else getattr(obs, 'planets', [])
    raw_fleets  = obs.get('fleets',  []) if isinstance(obs, dict) else getattr(obs, 'fleets',  [])
    ang_vel     = obs.get('angular_velocity', 0.0) if isinstance(obs, dict) else getattr(obs, 'angular_velocity', 0.0)
    step        = obs.get('step', 0) if isinstance(obs, dict) else getattr(obs, 'step', 0)
    comet_ids   = set(obs.get('comet_planet_ids', []) if isinstance(obs, dict) else getattr(obs, 'comet_planet_ids', []))

    n_planets = len(raw_planets)
    p_feat = np.zeros((MAX_PLANETS, PLANET_FEATURES), dtype=np.float32)
    p_mask = np.zeros((MAX_PLANETS,), dtype=np.float32)
    my_mask = np.zeros((MAX_PLANETS,), dtype=np.float32)
    for i, p in enumerate(raw_planets[:MAX_PLANETS]):
        pid, owner, x, y, radius, ships, prod = p
        dx, dy = x - CENTER, y - CENTER
        r = math.hypot(dx, dy)
        theta = math.atan2(dy, dx)
        is_static = 1.0 if r + radius >= ROTATION_LIMIT else 0.0
        is_comet = 1.0 if pid in comet_ids else 0.0
        # Owner one-hot in player-relative order: me, opp1, opp2_or_neutral, neutral
        # For 1v1: [me, opp, neutral, comet_pad]
        # For 4P: [me, opp_left, opp_right, opp_across] — but we'll just use me/opp/neutral/other for now
        ohot = [0.0, 0.0, 0.0, 0.0]
        if owner == player_id:
            ohot[0] = 1.0
            my_mask[i] = 1.0
        elif owner == -1:
            ohot[2] = 1.0
        else:
            ohot[1] = 1.0  # any non-self non-neutral
        p_feat[i] = [r/50.0, theta/math.pi, ang_vel/0.05 * (1.0 - is_static),
                     radius/3.0, ships/100.0, prod/5.0,
                     ohot[0], ohot[1], ohot[2], ohot[3],
                     is_static, is_comet]
        p_mask[i] = 1.0

    f_feat = np.zeros((MAX_FLEETS, FLEET_FEATURES), dtype=np.float32)
    f_mask = np.zeros((MAX_FLEETS,), dtype=np.float32)
    for i, f in enumerate(raw_fleets[:MAX_FLEETS]):
        fid, owner, x, y, angle, _, ships = f
        dx, dy = x - CENTER, y - CENTER
        r = math.hypot(dx, dy)
        theta = math.atan2(dy, dx)
        is_mine = 1.0 if owner == player_id else 0.0
        f_feat[i] = [r/50.0, theta/math.pi,
                     math.cos(angle), math.sin(angle),
                     ships/100.0, fleet_speed(ships)/MAX_SPEED,
                     is_mine, 1.0 - is_mine]
        f_mask[i] = 1.0

    # Global features
    my_ships = sum(p[5] for p in raw_planets if p[1] == player_id) + sum(f[6] for f in raw_fleets if f[1] == player_id)
    opp_ships = sum(p[5] for p in raw_planets if p[1] not in (-1, player_id)) + sum(f[6] for f in raw_fleets if f[1] != player_id and f[1] != -1)
    total_ships = max(1, my_ships + opp_ships)
    my_planets = sum(1 for p in raw_planets if p[1] == player_id)
    opp_planets = sum(1 for p in raw_planets if p[1] not in (-1, player_id))
    g_feat = np.array([
        step / TOTAL_STEPS,
        ang_vel / 0.05,
        my_ships / total_ships,
        opp_ships / total_ships,
        my_planets / 12.0,
        opp_planets / 12.0,
        1.0 if comet_ids else 0.0,
        1.0 if step > 100 and step < 400 else 0.0,
    ], dtype=np.float32)

    return p_feat, f_feat, g_feat, p_mask, f_mask, my_mask

## 3. Policy / Value network

Small set-transformer (~80K params). Per my-planet head outputs pointer attention over targets + send fraction logits.

In [ ]:
HIDDEN = 64
N_HEADS = 4
N_LAYERS = 2

class PlanetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.planet_proj = nn.Linear(PLANET_FEATURES, HIDDEN)
        self.fleet_proj = nn.Linear(FLEET_FEATURES, HIDDEN)
        self.global_proj = nn.Linear(GLOBAL_FEATURES, HIDDEN)
        # Transformer over planets only (fleets and global pooled separately)
        enc_layer = nn.TransformerEncoderLayer(d_model=HIDDEN, nhead=N_HEADS,
                                                dim_feedforward=HIDDEN*2,
                                                batch_first=True, dropout=0.0,
                                                activation='gelu')
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=N_LAYERS)

    def forward(self, p_feat, f_feat, g_feat, p_mask, f_mask):
        # p_feat: [B, MAX_PLANETS, PF]  p_mask: [B, MAX_PLANETS]
        ph = self.planet_proj(p_feat)  # [B, MP, H]
        # Transformer with key-padding mask (True where padded)
        pad_mask = (p_mask < 0.5)
        ph = self.transformer(ph, src_key_padding_mask=pad_mask)
        # Pool fleets (mean over present)
        fh = self.fleet_proj(f_feat)
        f_w = f_mask.unsqueeze(-1)
        fh_pooled = (fh * f_w).sum(dim=1) / (f_w.sum(dim=1) + 1e-6)
        # Pool planets (mean over present)
        p_w = p_mask.unsqueeze(-1)
        ph_pooled = (ph * p_w).sum(dim=1) / (p_w.sum(dim=1) + 1e-6)
        # Global head
        gh = self.global_proj(g_feat)  # [B, H]
        return ph, ph_pooled, fh_pooled, gh


class Actor(nn.Module):
    """Per my-planet head: target pointer (over all planets + NOOP) + send fraction.
    NOOP logit is computed from a learned 'noop query' attention."""
    def __init__(self):
        super().__init__()
        # Pointer attention: my_planet_hidden -> attention over all planet hidden states.
        self.query_proj = nn.Linear(HIDDEN, HIDDEN)
        self.key_proj = nn.Linear(HIDDEN, HIDDEN)
        self.noop_logit = nn.Parameter(torch.zeros(1))
        self.send_head = nn.Sequential(
            nn.Linear(HIDDEN, HIDDEN), nn.GELU(),
            nn.Linear(HIDDEN, N_SEND_BINS),
        )

    def forward(self, ph, p_mask, my_mask):
        # ph: [B, MP, H]  my_mask: [B, MP] (1 where my planet)
        B, MP, H = ph.shape
        Q = self.query_proj(ph)  # [B, MP, H]  — every planet has a query
        K = self.key_proj(ph)    # [B, MP, H]
        attn = torch.einsum('bmh,bnh->bmn', Q, K) / math.sqrt(H)  # [B, MP, MP]
        # Mask: cannot target self (diagonal) and cannot target padded planets.
        eye = torch.eye(MP, device=attn.device).unsqueeze(0)  # [1, MP, MP]
        attn = attn.masked_fill(eye.bool(), -1e9)
        pad = (p_mask < 0.5).unsqueeze(1).expand(-1, MP, -1)  # [B, MP, MP]
        attn = attn.masked_fill(pad, -1e9)
        # Concatenate NOOP logit (broadcast per (B, MP))
        noop = self.noop_logit.expand(B, MP, 1)
        target_logits = torch.cat([attn, noop], dim=-1)  # [B, MP, MP+1]
        # Send fraction
        send_logits = self.send_head(ph)  # [B, MP, N_SEND_BINS]
        return target_logits, send_logits


class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(HIDDEN*3, HIDDEN), nn.GELU(),
            nn.Linear(HIDDEN, 1),
        )

    def forward(self, ph_pooled, fh_pooled, gh):
        z = torch.cat([ph_pooled, fh_pooled, gh], dim=-1)  # [B, 3H]
        return self.head(z).squeeze(-1)  # [B]


class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = PlanetEncoder()
        self.actor = Actor()
        self.critic = Critic()

    def forward(self, p_feat, f_feat, g_feat, p_mask, f_mask, my_mask):
        ph, php, fhp, gh = self.enc(p_feat, f_feat, g_feat, p_mask, f_mask)
        target_logits, send_logits = self.actor(ph, p_mask, my_mask)
        value = self.critic(php, fhp, gh)
        return target_logits, send_logits, value


net = ActorCritic().to(DEVICE)
n_params = sum(p.numel() for p in net.parameters())
print(f'net params: {n_params:,}')

## 4. Action sampling + applying to engine

Each turn, every my-planet independently picks a target (or NOOP) and a send-fraction bin. Convert to engine `[from_id, angle, num_ships]` moves.

In [ ]:
def sample_actions(net, obs, player_id, deterministic=False):
    """Returns (moves_list, action_record). action_record captures everything needed for PPO update."""
    p_feat, f_feat, g_feat, p_mask, f_mask, my_mask = encode_state(obs, player_id)
    if my_mask.sum() < 0.5:
        # No my planets — nothing to do (game over for us)
        return [], None
    pt = lambda a: torch.from_numpy(a).unsqueeze(0).to(DEVICE)
    p_t, f_t, g_t, pm_t, fm_t, mm_t = pt(p_feat), pt(f_feat), pt(g_feat), pt(p_mask), pt(f_mask), pt(my_mask)
    with torch.no_grad():
        target_logits, send_logits, value = net(p_t, f_t, g_t, pm_t, fm_t, mm_t)
    target_logits = target_logits.squeeze(0)  # [MP, MP+1]
    send_logits = send_logits.squeeze(0)      # [MP, N_SEND_BINS]
    value = value.item()
    target_dist = torch.distributions.Categorical(logits=target_logits)
    send_dist = torch.distributions.Categorical(logits=send_logits)
    if deterministic:
        target_a = target_logits.argmax(dim=-1)
        send_a = send_logits.argmax(dim=-1)
    else:
        target_a = target_dist.sample()
        send_a = send_dist.sample()
    target_logp = target_dist.log_prob(target_a)
    send_logp = send_dist.log_prob(send_a)
    # Build moves only for my planets (mask others)
    raw_planets = obs.get('planets', []) if isinstance(obs, dict) else getattr(obs, 'planets', [])
    moves = []
    NOOP_IDX = MAX_PLANETS
    for i in range(min(MAX_PLANETS, len(raw_planets))):
        if my_mask[i] < 0.5: continue
        tgt = int(target_a[i].item())
        if tgt == NOOP_IDX or tgt >= len(raw_planets): continue
        if tgt == i: continue  # masked but defensive
        src = raw_planets[i]
        dst = raw_planets[tgt]
        # send fraction
        frac = SEND_FRACTIONS[int(send_a[i].item())]
        ships = int(src[5] * frac)
        if ships < 1: continue
        angle = math.atan2(dst[3] - src[3], dst[2] - src[2])
        moves.append([src[0], angle, ships])
    record = {
        'p_feat': p_feat, 'f_feat': f_feat, 'g_feat': g_feat,
        'p_mask': p_mask, 'f_mask': f_mask, 'my_mask': my_mask,
        'target_a': target_a.cpu().numpy(),
        'send_a': send_a.cpu().numpy(),
        'target_logp': target_logp.cpu().numpy(),
        'send_logp': send_logp.cpu().numpy(),
        'value': value,
    }
    return moves, record

## 5. Self-play rollout collector

Plays a full game with both seats using our net (one is the current policy, other is a frozen league snapshot). Returns transition records for the LEARNING seat only.

In [ ]:
def make_agent_from_net(net_ref, deterministic=False):
    """Return an agent function suitable for env.run([...]). Records ignored."""
    def agent(obs, config=None):
        player_id = obs.get('player', 0) if isinstance(obs, dict) else getattr(obs, 'player', 0)
        moves, _ = sample_actions(net_ref, obs, player_id, deterministic=deterministic)
        return moves
    return agent


def collect_rollout(learner_net, opponent_net, max_steps=300, seed=None):
    """Play one game. learner_net is in seat 0 (random side flip for symmetry).
    Returns list of step records + final reward for the learner."""
    flip = random.random() < 0.5  # 50% chance learner is in seat 1
    cfg = {'episodeSteps': max_steps, 'actTimeout': 30}
    if seed is not None: cfg['seed'] = seed
    env = make('orbit_wars', configuration=cfg, debug=False)
    state = env.reset(num_agents=2)
    # We need step-by-step access to record states for the learner only.
    learner_seat = 1 if flip else 0
    learner_records = []
    steps = 0
    while not env.done and steps < max_steps:
        actions = []
        for i in range(2):
            obs_i = state[i].observation
            if i == learner_seat:
                moves, rec = sample_actions(learner_net, obs_i, i, deterministic=False)
                if rec is not None:
                    learner_records.append({**rec, 'step': steps})
                actions.append(moves)
            else:
                moves, _ = sample_actions(opponent_net, obs_i, i, deterministic=True)
                actions.append(moves)
        state = env.step(actions)
        steps += 1
    final_reward = state[learner_seat].reward if state[learner_seat].reward is not None else 0
    return learner_records, final_reward, steps

## 6. PPO update

Standard PPO: clipped surrogate (ε=0.2) + value loss + small entropy bonus. GAE for advantages.

In [ ]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """Generalized advantage estimation. rewards: [T], values: [T+1] (bootstrap last). Returns adv [T], returns [T]."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t+1] - values[t]
        gae = delta + gamma * lam * gae
        adv[t] = gae
    returns = adv + np.array(values[:-1], dtype=np.float32)
    return adv, returns


def ppo_update(net, optimizer, batch, clip_eps=0.2, vf_coef=0.5, ent_coef=0.01, n_epochs=4, mb_size=64):
    """batch is dict of stacked tensors on DEVICE.
    Robust to: degenerate minibatch (size 1), NaN/Inf in loss (skip update), tiny adv std."""
    N = batch['p_feat'].shape[0]
    idx = np.arange(N)
    stats = {'policy_loss': 0, 'value_loss': 0, 'entropy': 0, 'kl': 0, 'n': 0, 'skipped': 0}
    for _ in range(n_epochs):
        np.random.shuffle(idx)
        for start in range(0, N, mb_size):
            mb = idx[start:start+mb_size]
            # Skip degenerate minibatches (size 1 → std() returns NaN with unbiased=True).
            if len(mb) < 2:
                stats['skipped'] += 1
                continue
            p_t = batch['p_feat'][mb]; f_t = batch['f_feat'][mb]; g_t = batch['g_feat'][mb]
            pm_t = batch['p_mask'][mb]; fm_t = batch['f_mask'][mb]; mm_t = batch['my_mask'][mb]
            target_a = batch['target_a'][mb]; send_a = batch['send_a'][mb]
            old_t_logp = batch['target_logp'][mb]; old_s_logp = batch['send_logp'][mb]
            adv = batch['adv'][mb]; ret = batch['ret'][mb]
            target_logits, send_logits, value = net(p_t, f_t, g_t, pm_t, fm_t, mm_t)
            # Sanity: if forward already produced NaNs, weights are corrupted — skip and let user notice.
            if not torch.isfinite(target_logits).all() or not torch.isfinite(send_logits).all() or not torch.isfinite(value).all():
                stats['skipped'] += 1
                continue
            t_dist = torch.distributions.Categorical(logits=target_logits)
            s_dist = torch.distributions.Categorical(logits=send_logits)
            new_t_logp = t_dist.log_prob(target_a)  # [B, MP]
            new_s_logp = s_dist.log_prob(send_a)
            # Only count log-probs for actual my-planet actions (mask)
            t_lp_sum = (new_t_logp * mm_t).sum(dim=1)
            s_lp_sum = (new_s_logp * mm_t).sum(dim=1)
            old_t_lp_sum = (old_t_logp * mm_t).sum(dim=1)
            old_s_lp_sum = (old_s_logp * mm_t).sum(dim=1)
            new_lp = t_lp_sum + s_lp_sum
            old_lp = old_t_lp_sum + old_s_lp_sum
            # Advantage normalization — guard against degenerate cases.
            adv_mean = adv.mean()
            # unbiased=False avoids NaN when len(adv)==1; clamp to floor.
            adv_std = adv.std(unbiased=False)
            if not torch.isfinite(adv_std) or adv_std < 1e-6:
                adv_norm = adv - adv_mean
            else:
                adv_norm = (adv - adv_mean) / (adv_std + 1e-8)
            ratio = torch.exp(new_lp - old_lp)
            surr1 = ratio * adv_norm
            surr2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * adv_norm
            pol_loss = -torch.min(surr1, surr2).mean()
            val_loss = F.mse_loss(value, ret)
            ent = (t_dist.entropy() * mm_t).sum(dim=1).mean() + (s_dist.entropy() * mm_t).sum(dim=1).mean()
            loss = pol_loss + vf_coef * val_loss - ent_coef * ent
            # Final safety: never propagate NaN into the optimizer.
            if not torch.isfinite(loss):
                stats['skipped'] += 1
                continue
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            optimizer.step()
            with torch.no_grad():
                kl = (old_lp - new_lp).mean().item()
            stats['policy_loss'] += pol_loss.item()
            stats['value_loss'] += val_loss.item()
            stats['entropy'] += ent.item()
            stats['kl'] += kl
            stats['n'] += 1
    for k in stats:
        if k not in ('n', 'skipped'): stats[k] /= max(1, stats['n'])
    return stats


## 7. Checkpoint management

Save only `last.pt` and `best.pt`. Delete other .pt files to stay under 20 GB.

In [ ]:
def save_ckpt(path, net, optimizer, league, total_steps, total_games, best_eval_winrate, rng_state):
    torch.save({
        'net': net.state_dict(),
        'optimizer': optimizer.state_dict(),
        'league': [snap_state for snap_state in league],   # list of state_dicts
        'total_steps': total_steps,
        'total_games': total_games,
        'best_eval_winrate': best_eval_winrate,
        'rng_state': rng_state,
    }, path)
    # Clean up any rogue .pt files
    for p in glob.glob(os.path.join(WORKING_DIR, '*.pt')):
        if os.path.basename(p) not in ('last.pt', 'best.pt'):
            try: os.remove(p)
            except: pass


def load_ckpt(path, net, optimizer):
    if not os.path.exists(path): return None
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    net.load_state_dict(ck['net'])
    optimizer.load_state_dict(ck['optimizer'])
    return ck


def log_metrics(d):
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps(d) + '\n')

## 8. Training loop

Curriculum: first ~200 games vs `random_agent` (easy starting signal), then self-play vs league.  
Eval every 50 iterations against `starter` agent — best-by-winrate snapshot becomes `best.pt`.

In [ ]:
from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent, starter_agent

# ===== Hyperparams v2 — reward shaping + BC warm-start + lower entropy =====
GAMES_PER_ITER = 12          # was 8 — less noisy updates
MAX_STEPS_PER_GAME = 300
PPO_EPOCHS = 4
PPO_MB_SIZE = 64
LR = 5e-4                    # was 3e-4 — help escape uniform-policy plateau
ENT_COEF = 0.003             # was 0.01 — we want the policy to actually commit
REWARD_SHAPING_COEF = 0.01   # per-step (my_ship_delta - opp_ship_delta) / 50 × this
REWARD_SHAPING_DECAY = 0.999 # anneal shaping toward sparse over time
LEAGUE_SIZE = 8
LEAGUE_SNAPSHOT_EVERY = 25
EVAL_EVERY = 25
EVAL_GAMES = 16
CKPT_EVERY = 25
CURRICULUM_RANDOM_ITERS = 25
TIME_BUDGET_HRS = 11.5

BC_CKPT = os.path.join(WORKING_DIR, 'bc_init.pt')

optimizer = torch.optim.Adam(net.parameters(), lr=LR)

# Resume: last.pt has priority, then bc_init.pt as warm-start.
ck = load_ckpt(LAST_CKPT, net, optimizer)
if ck is not None:
    # Override LR to the new value (loaded optimizer state may carry the old LR).
    for g in optimizer.param_groups:
        g['lr'] = LR
    total_steps = ck.get('total_steps', 0)
    total_games = ck.get('total_games', 0)
    best_eval_winrate = ck.get('best_eval_winrate', 0.0)
    league_states = ck.get('league', [])
    print(f'resumed from last.pt: steps={total_steps:,} games={total_games:,} best_eval={best_eval_winrate:.2%}  (LR forced to {LR})')
elif os.path.exists(BC_CKPT):
    bc = torch.load(BC_CKPT, map_location=DEVICE, weights_only=False)
    net.load_state_dict(bc['net'])
    total_steps = 0; total_games = 0; best_eval_winrate = 0.0; league_states = []
    val_loss = bc.get('val_loss', float('nan'))
    print(f'no last.pt; warm-started from bc_init.pt (val_loss={val_loss:.3f})')
else:
    total_steps = 0; total_games = 0; best_eval_winrate = 0.0; league_states = []
    print('starting from scratch (no bc_init.pt, no last.pt)')


def league_pick_opponent(league_states):
    if not league_states:
        return None
    weights = [1 + 2*i for i in range(len(league_states))]
    return random.choices(league_states, weights=weights, k=1)[0]

def reify_opponent_net(state_dict):
    op = ActorCritic().to(DEVICE)
    op.load_state_dict(state_dict); op.eval()
    return op

class CallableAgent:
    def __init__(self, fn): self.fn = fn
    def __call__(self, obs, config=None): return self.fn(obs)

random_baseline = CallableAgent(random_agent)
starter_baseline = CallableAgent(starter_agent)


def _ship_totals(obs, player_id):
    """Return (my_total_ships, max_opp_total_ships) for reward shaping."""
    raw_planets = obs.get('planets', []) if isinstance(obs, dict) else getattr(obs, 'planets', [])
    raw_fleets  = obs.get('fleets',  []) if isinstance(obs, dict) else getattr(obs, 'fleets',  [])
    my = sum(p[5] for p in raw_planets if p[1] == player_id) + sum(f[6] for f in raw_fleets if f[1] == player_id)
    per_opp = defaultdict(int)
    for p in raw_planets:
        if p[1] not in (-1, player_id): per_opp[p[1]] += p[5]
    for f in raw_fleets:
        if f[1] not in (-1, player_id): per_opp[f[1]] += f[6]
    opp = max(per_opp.values()) if per_opp else 0
    return my, opp


def collect_rollout_against(opponent_agent, max_steps, seed=None):
    """opponent_agent is a callable taking obs (and optional config).
    Returns (records, final_reward, n_steps). Each record now also carries
    'my_ships' / 'opp_ships' totals at that step for reward shaping."""
    flip = random.random() < 0.5
    cfg = {'episodeSteps': max_steps, 'actTimeout': 30}
    if seed is not None: cfg['seed'] = seed
    env = make('orbit_wars', configuration=cfg, debug=False)
    state = env.reset(num_agents=2)
    learner_seat = 1 if flip else 0
    learner_records = []
    steps = 0
    while not env.done and steps < max_steps:
        actions = []
        for i in range(2):
            obs_i = state[i].observation
            if i == learner_seat:
                moves, rec = sample_actions(net, obs_i, i, deterministic=False)
                if rec is not None:
                    my_s, opp_s = _ship_totals(obs_i, i)
                    rec['my_ships'] = my_s
                    rec['opp_ships'] = opp_s
                    learner_records.append({**rec, 'step': steps})
                actions.append(moves)
            else:
                moves = opponent_agent(obs_i)
                actions.append(moves)
        state = env.step(actions)
        steps += 1
    final_reward = state[learner_seat].reward if state[learner_seat].reward is not None else 0
    return learner_records, final_reward, steps


def evaluate_vs(opponent_agent, n_games=EVAL_GAMES):
    wins = 0
    for g in range(n_games):
        flip = g % 2 == 1
        cfg = {'episodeSteps': 500, 'actTimeout': 30, 'seed': 9000 + g}
        env = make('orbit_wars', configuration=cfg, debug=False)
        learner = make_agent_from_net(net, deterministic=True)
        agents = [opponent_agent, learner] if flip else [learner, opponent_agent]
        state = env.run(agents)
        learner_seat = 1 if flip else 0
        r = state[-1][learner_seat].reward
        if r is not None and r > 0: wins += 1
    return wins / n_games


def shaped_rewards(recs, terminal_reward, coef):
    """Per-step reward = coef × (Δmy_ships − Δopp_ships) / 50, plus terminal at last step."""
    rewards = [0.0] * len(recs)
    for i in range(1, len(recs)):
        dm = recs[i]['my_ships'] - recs[i-1]['my_ships']
        do = recs[i]['opp_ships'] - recs[i-1]['opp_ships']
        rewards[i] = coef * (dm - do) / 50.0
    if recs:
        rewards[-1] += float(terminal_reward)  # add sparse terminal
    return rewards


# Main loop
t_start = time.time()
iter_idx = 0
shape_coef = REWARD_SHAPING_COEF
while True:
    elapsed_hrs = (time.time() - t_start) / 3600
    if elapsed_hrs > TIME_BUDGET_HRS:
        print(f'⏰ Time budget reached ({elapsed_hrs:.1f}h). Final save and stop.')
        save_ckpt(LAST_CKPT, net, optimizer, league_states, total_steps, total_games, best_eval_winrate, random.getstate())
        break

    iter_idx += 1
    shape_coef = REWARD_SHAPING_COEF * (REWARD_SHAPING_DECAY ** iter_idx)

    using_random = (iter_idx <= CURRICULUM_RANDOM_ITERS and total_games < 200)
    if using_random or not league_states:
        opp = random_baseline; opp_tag = 'random'
    else:
        opp_state = league_pick_opponent(league_states)
        opp_net = reify_opponent_net(opp_state)
        opp = make_agent_from_net(opp_net, deterministic=True)
        opp_tag = f'league({len(league_states)})'

    all_records = []
    rewards_per_game = []
    for g in range(GAMES_PER_ITER):
        recs, final_r, n_steps = collect_rollout_against(opp, MAX_STEPS_PER_GAME, seed=None)
        rewards = shaped_rewards(recs, final_r, shape_coef)
        values = [r['value'] for r in recs] + [0.0]
        adv, ret = compute_gae(rewards, values)
        for i, r in enumerate(recs):
            r['adv'] = adv[i]; r['ret'] = ret[i]
        all_records.extend(recs)
        rewards_per_game.append(final_r)
        total_steps += n_steps
        total_games += 1

    if not all_records:
        continue

    def stack_field(records, field, dtype=torch.float32):
        arr = np.stack([r[field] for r in records], axis=0)
        return torch.from_numpy(arr).to(DEVICE).to(dtype)
    batch = {
        'p_feat': stack_field(all_records, 'p_feat'),
        'f_feat': stack_field(all_records, 'f_feat'),
        'g_feat': stack_field(all_records, 'g_feat'),
        'p_mask': stack_field(all_records, 'p_mask'),
        'f_mask': stack_field(all_records, 'f_mask'),
        'my_mask': stack_field(all_records, 'my_mask'),
        'target_a': stack_field(all_records, 'target_a', dtype=torch.long),
        'send_a': stack_field(all_records, 'send_a', dtype=torch.long),
        'target_logp': stack_field(all_records, 'target_logp'),
        'send_logp': stack_field(all_records, 'send_logp'),
        'adv': stack_field(all_records, 'adv'),
        'ret': stack_field(all_records, 'ret'),
    }

    stats = ppo_update(net, optimizer, batch, n_epochs=PPO_EPOCHS, mb_size=PPO_MB_SIZE, ent_coef=ENT_COEF)
    wr_vs_opp = sum(1 for r in rewards_per_game if r > 0) / max(1, len(rewards_per_game))
    metrics = {
        'iter': iter_idx, 'total_games': total_games, 'total_steps': total_steps,
        'opp': opp_tag, 'wr_vs_opp': wr_vs_opp,
        'pol_loss': stats['policy_loss'], 'val_loss': stats['value_loss'],
        'entropy': stats['entropy'], 'kl': stats['kl'],
        'shape_coef': shape_coef, 'elapsed_hrs': elapsed_hrs,
    }
    log_metrics(metrics)
    print(f"iter {iter_idx:4d}  games={total_games:5d}  vs {opp_tag:14s}  "
          f"wr={wr_vs_opp:.2f}  pol={stats['policy_loss']:.3f}  val={stats['value_loss']:.3f}  "
          f"ent={stats['entropy']:.2f}  kl={stats['kl']:+.3f}  shape={shape_coef:.4f}  [{elapsed_hrs:.2f}h]")

    if iter_idx % LEAGUE_SNAPSHOT_EVERY == 0:
        snap = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        league_states.append(snap)
        if len(league_states) > LEAGUE_SIZE:
            league_states.pop(0)

    if iter_idx % EVAL_EVERY == 0:
        eval_wr = evaluate_vs(starter_baseline, n_games=EVAL_GAMES)
        print(f'  ↳ eval vs starter ({EVAL_GAMES} games): {eval_wr:.2%}')
        log_metrics({'iter': iter_idx, 'event': 'eval', 'eval_wr_vs_starter': eval_wr})
        if eval_wr > best_eval_winrate:
            best_eval_winrate = eval_wr
            save_ckpt(BEST_CKPT, net, optimizer, league_states, total_steps, total_games, best_eval_winrate, random.getstate())
            print(f'  ★ new best ({best_eval_winrate:.2%}) → best.pt')

    if iter_idx % CKPT_EVERY == 0:
        save_ckpt(LAST_CKPT, net, optimizer, league_states, total_steps, total_games, best_eval_winrate, random.getstate())
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


## 9. Export to a torch-free `main.py`

Kaggle submissions allow torch, but bundling it (~2 GB) is heavy. Our net is small (~80 K params) — easy to do inference with NumPy only. This cell loads `best.pt` and writes a self-contained `main.py` that runs the policy without any ML framework.

In [ ]:
import textwrap

ck = torch.load(BEST_CKPT, map_location='cpu', weights_only=False)
state = ck['net']
# Save weights as compressed numpy archive
weights_path = os.path.join(WORKING_DIR, 'weights.npz')
np.savez_compressed(weights_path,
    **{k: v.numpy() for k, v in state.items()},
)
print(f'wrote {weights_path}  ({os.path.getsize(weights_path)/1024:.1f} KB)')

# Write the inference main.py. Uses numpy only.
inference_src = '''
# Orbit Wars — PPO-trained agent (numpy-only inference)
import os, math, numpy as np

BOARD = 100.0; CENTER = 50.0; SUN_R = 10.0; ROTATION_LIMIT = 50.0; MAX_SPEED = 6.0
PLANET_FEATURES = ''' + str(PLANET_FEATURES) + '''
FLEET_FEATURES = ''' + str(FLEET_FEATURES) + '''
GLOBAL_FEATURES = ''' + str(GLOBAL_FEATURES) + '''
MAX_PLANETS = ''' + str(MAX_PLANETS) + '''
MAX_FLEETS = ''' + str(MAX_FLEETS) + '''
HIDDEN = ''' + str(HIDDEN) + '''
N_HEADS = ''' + str(N_HEADS) + '''
N_LAYERS = ''' + str(N_LAYERS) + '''
SEND_FRACTIONS = ''' + str(SEND_FRACTIONS) + '''

WEIGHTS = None
def _load_weights():
    global WEIGHTS
    if WEIGHTS is None:
        path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "weights.npz")
        with np.load(path) as f:
            WEIGHTS = {k: f[k] for k in f.files}
    return WEIGHTS

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def layer_norm(x, gamma, beta, eps=1e-5):
    m = x.mean(-1, keepdims=True); v = x.var(-1, keepdims=True)
    return gamma * (x - m) / np.sqrt(v + eps) + beta

def linear(x, w, b):
    return x @ w.T + b

def attention(q, k, v, mask=None):
    # q,k,v: [..., L, D]
    scores = q @ k.swapaxes(-1, -2) / np.sqrt(q.shape[-1])
    if mask is not None: scores = np.where(mask, -1e9, scores)
    a = softmax(scores, axis=-1)
    return a @ v

def fleet_speed(ships):
    if ships <= 1: return 1.0
    r = min(1.0, max(0.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (r ** 1.5)

def encode_state(obs, player_id):
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else getattr(obs, "planets", [])
    raw_fleets  = obs.get("fleets",  []) if isinstance(obs, dict) else getattr(obs, "fleets",  [])
    ang_vel     = obs.get("angular_velocity", 0.0) if isinstance(obs, dict) else getattr(obs, "angular_velocity", 0.0)
    step        = obs.get("step", 0) if isinstance(obs, dict) else getattr(obs, "step", 0)
    comet_ids   = set(obs.get("comet_planet_ids", []) if isinstance(obs, dict) else getattr(obs, "comet_planet_ids", []))
    p_feat = np.zeros((MAX_PLANETS, PLANET_FEATURES), dtype=np.float32)
    p_mask = np.zeros((MAX_PLANETS,), dtype=np.float32)
    my_mask = np.zeros((MAX_PLANETS,), dtype=np.float32)
    for i, p in enumerate(raw_planets[:MAX_PLANETS]):
        pid, owner, x, y, radius, ships, prod = p
        dx, dy = x - CENTER, y - CENTER
        r = math.hypot(dx, dy); theta = math.atan2(dy, dx)
        is_static = 1.0 if r + radius >= ROTATION_LIMIT else 0.0
        is_comet = 1.0 if pid in comet_ids else 0.0
        ohot = [0.0, 0.0, 0.0, 0.0]
        if owner == player_id: ohot[0] = 1.0; my_mask[i] = 1.0
        elif owner == -1: ohot[2] = 1.0
        else: ohot[1] = 1.0
        p_feat[i] = [r/50.0, theta/math.pi, ang_vel/0.05 * (1.0 - is_static),
                     radius/3.0, ships/100.0, prod/5.0,
                     ohot[0], ohot[1], ohot[2], ohot[3],
                     is_static, is_comet]
        p_mask[i] = 1.0
    f_feat = np.zeros((MAX_FLEETS, FLEET_FEATURES), dtype=np.float32)
    f_mask = np.zeros((MAX_FLEETS,), dtype=np.float32)
    for i, f in enumerate(raw_fleets[:MAX_FLEETS]):
        fid, owner, x, y, angle, _, ships = f
        dx, dy = x - CENTER, y - CENTER
        r = math.hypot(dx, dy); theta = math.atan2(dy, dx)
        is_mine = 1.0 if owner == player_id else 0.0
        f_feat[i] = [r/50.0, theta/math.pi, math.cos(angle), math.sin(angle),
                     ships/100.0, fleet_speed(ships)/MAX_SPEED, is_mine, 1.0 - is_mine]
        f_mask[i] = 1.0
    my_ships = sum(p[5] for p in raw_planets if p[1] == player_id) + sum(f[6] for f in raw_fleets if f[1] == player_id)
    opp_ships = sum(p[5] for p in raw_planets if p[1] not in (-1, player_id)) + sum(f[6] for f in raw_fleets if f[1] not in (-1, player_id))
    total_ships = max(1, my_ships + opp_ships)
    my_planets = sum(1 for p in raw_planets if p[1] == player_id)
    opp_planets = sum(1 for p in raw_planets if p[1] not in (-1, player_id))
    g_feat = np.array([
        step / 500.0, ang_vel / 0.05,
        my_ships / total_ships, opp_ships / total_ships,
        my_planets / 12.0, opp_planets / 12.0,
        1.0 if comet_ids else 0.0, 1.0 if step > 100 and step < 400 else 0.0,
    ], dtype=np.float32)
    return p_feat, f_feat, g_feat, p_mask, f_mask, my_mask

def forward(obs, player_id):
    W = _load_weights()
    p_feat, f_feat, g_feat, p_mask, f_mask, my_mask = encode_state(obs, player_id)
    # Linear projections
    ph = linear(p_feat, W["enc.planet_proj.weight"], W["enc.planet_proj.bias"])  # [MP, H]
    fh = linear(f_feat, W["enc.fleet_proj.weight"], W["enc.fleet_proj.bias"])
    gh = linear(g_feat, W["enc.global_proj.weight"], W["enc.global_proj.bias"])
    # Transformer over planets (N_LAYERS, all with same head structure)
    pad_mask = (p_mask < 0.5)  # [MP]
    for layer in range(N_LAYERS):
        prefix = f"enc.transformer.layers.{layer}."
        # Multi-head self-attention
        Wqkv = W[prefix+"self_attn.in_proj_weight"]; bqkv = W[prefix+"self_attn.in_proj_bias"]
        qkv = linear(ph, Wqkv, bqkv)  # [MP, 3H]
        q, k, v = np.split(qkv, 3, axis=-1)
        # Reshape for heads
        head_dim = HIDDEN // N_HEADS
        q = q.reshape(-1, N_HEADS, head_dim).transpose(1, 0, 2)  # [H, MP, hd]
        k = k.reshape(-1, N_HEADS, head_dim).transpose(1, 0, 2)
        v = v.reshape(-1, N_HEADS, head_dim).transpose(1, 0, 2)
        # Per-head attention with padding mask
        mask_h = np.broadcast_to(pad_mask, (N_HEADS, p_mask.shape[0], p_mask.shape[0]))
        scores = q @ k.swapaxes(-1, -2) / np.sqrt(head_dim)
        scores = np.where(mask_h, -1e9, scores)
        a = softmax(scores, axis=-1)
        out = a @ v  # [H, MP, hd]
        out = out.transpose(1, 0, 2).reshape(-1, HIDDEN)  # [MP, H]
        # Out proj
        out = linear(out, W[prefix+"self_attn.out_proj.weight"], W[prefix+"self_attn.out_proj.bias"])
        ph = ph + out
        ph = layer_norm(ph, W[prefix+"norm1.weight"], W[prefix+"norm1.bias"])
        # FFN
        ff = linear(ph, W[prefix+"linear1.weight"], W[prefix+"linear1.bias"])
        ff = gelu(ff)
        ff = linear(ff, W[prefix+"linear2.weight"], W[prefix+"linear2.bias"])
        ph = ph + ff
        ph = layer_norm(ph, W[prefix+"norm2.weight"], W[prefix+"norm2.bias"])
    # Actor head: pointer attention
    Q = linear(ph, W["actor.query_proj.weight"], W["actor.query_proj.bias"])
    K = linear(ph, W["actor.key_proj.weight"], W["actor.key_proj.bias"])
    target_scores = Q @ K.T / np.sqrt(HIDDEN)  # [MP, MP]
    # Mask self and padded
    target_scores = target_scores - 1e9 * np.eye(MAX_PLANETS)
    target_scores = np.where(pad_mask[None, :], -1e9, target_scores)
    noop_logit = W["actor.noop_logit"]
    target_logits = np.concatenate([target_scores, np.broadcast_to(noop_logit, (MAX_PLANETS, 1))], axis=-1)
    # Send head
    s1 = gelu(linear(ph, W["actor.send_head.0.weight"], W["actor.send_head.0.bias"]))
    send_logits = linear(s1, W["actor.send_head.2.weight"], W["actor.send_head.2.bias"])
    return target_logits, send_logits, my_mask

def agent(obs, config=None):
    player_id = obs.get("player", 0) if isinstance(obs, dict) else getattr(obs, "player", 0)
    target_logits, send_logits, my_mask = forward(obs, player_id)
    # Deterministic: argmax
    target_a = target_logits.argmax(axis=-1)
    send_a = send_logits.argmax(axis=-1)
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else getattr(obs, "planets", [])
    moves = []
    NOOP_IDX = MAX_PLANETS
    for i in range(min(MAX_PLANETS, len(raw_planets))):
        if my_mask[i] < 0.5: continue
        tgt = int(target_a[i])
        if tgt == NOOP_IDX or tgt >= len(raw_planets) or tgt == i: continue
        src = raw_planets[i]; dst = raw_planets[tgt]
        frac = SEND_FRACTIONS[int(send_a[i])]
        ships = int(src[5] * frac)
        if ships < 1: continue
        angle = math.atan2(dst[3] - src[3], dst[2] - src[2])
        moves.append([src[0], angle, ships])
    return moves
'''
main_path = os.path.join(WORKING_DIR, 'main_rl.py')
with open(main_path, 'w') as f:
    f.write(inference_src.lstrip('\n'))
print(f'wrote {main_path}')
print('\nTo submit: bundle main_rl.py + weights.npz into a tar.gz and submit:')
print('  tar -czf submission.tar.gz -C /kaggle/working main_rl.py weights.npz')
print('  # Or rename main_rl.py to main.py before tarring.')
print('  # Or download main_rl.py and weights.npz locally and submit from there.')

## 10. Sanity check — load weights from .npz and verify inference works

In [ ]:
# Verify the exported inference matches the torch model on a single obs.
env = make('orbit_wars', configuration={'episodeSteps': 50, 'actTimeout': 30, 'seed': 1}, debug=False)
state = env.reset(num_agents=2)
obs0 = state[0].observation
# Torch inference
torch_moves, _ = sample_actions(net, obs0, 0, deterministic=True)
print('torch moves:', torch_moves[:3])
# Run the exported numpy inference
import importlib.util
spec = importlib.util.spec_from_file_location('main_rl', main_path)
m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
np_moves = m.agent(obs0)
print('numpy moves:', np_moves[:3])
print('match (top action):', torch_moves[:1] == np_moves[:1] if torch_moves and np_moves else (not torch_moves and not np_moves))